In [ ]:
# 1. Uninstall conflicting packages
!pip uninstall -y unsloth unsloth-zoo transformers tokenizers

# 2. Install unsloth with all dependencies (this will get the right versions)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 3. Restart runtime
# Go to: Runtime → Restart runtime (or Ctrl+M)

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-3gddjgfq/unsloth_208a57f39af740e8a3c72a794f36e29a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-3gddjgfq/unsloth_208a57f39af740e8a3c72a794f36e29a
  Resolved https://github.com/unslothai/unsloth.git to commit 33b0343ec56595d4e7d7cdd25f173207ebf991b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached unsloth_zoo-2026.1.2-py3-none-any.whl.metadata (32 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached trl-0.24.0-py3-none-any.whl.metadata (11 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached unsloth_zoo-2026.1.2-py3-none-any.whl (295 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.8 MB/s eta 0:00:00
Using cached trl-0.24.0-py3-none-any.whl (423 kB)
  Created wheel for unsloth: fi

In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
df = pd.read_json("/content/synthetic_training_data.jsonl",lines=True)
print(f" Loaded {len(df)} samples")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst sample:")
print(df.iloc[0])



✅ Loaded 1364 samples
Columns: ['instruction', 'input', 'output']

First sample:
instruction    Analyze the following customer review and prov...
input                                                           
output          Profanity: yes\nSentiment: negative\nRewritten: 
Name: 0, dtype: object


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from datasets import Dataset
import pandas as pd
import torch

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="meta-llama/Meta-Llama-3-8B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print(" Model loaded successfully!")

# Continue with rest of training code...

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.1.2: Fast Llama patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

 Model loaded successfully!


In [ ]:
# ============================================================================
# STEP 3: Apply LoRA
# ============================================================================

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

Unsloth 2026.1.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 41,943,040 || all params: 8,072,204,288 || trainable%: 0.5196


In [ ]:
# ============================================================================
# STEP 4: Format Your Dataset for LFM
# ============================================================================

# LFM prompt template
lfm_prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{}<|eot_id|>"""

def format_prompt(sample):
    """Format each sample for LFM instruction following"""
    instruction = sample["instruction"]
    output = sample["output"]

    # Combine instruction and output into training text
    text = lfm_prompt.format(instruction, output)

    return {"text": text}

# Convert to HuggingFace Dataset
dataset = Dataset.from_pandas(df)

# Apply formatting
dataset = dataset.map(format_prompt, remove_columns=df.columns.tolist())

# Split into train/validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)

print(f"\n Dataset formatted:")
print(f"   Training: {len(dataset['train'])} samples")
print(f"   Validation: {len(dataset['test'])} samples")

# Preview formatted sample
print(f"\n Sample formatted text:")
print(dataset['train'][0]['text'] "...")

Map:   0%|          | 0/1364 [00:00<?, ? examples/s]


 Dataset formatted:
   Training: 1227 samples
   Validation: 137 samples

 Sample formatted text:
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)
3. If profanity is present, rewrite it in a polite, professional manner while preserving the meaning and sentiment.

Review: AMAZON SUCKS! THEY DONT EVEN SEND WHAT YOU ORDER THEY SEND SUBSTITUTION CRAP THEY CANT SELL!

Did NOT get what I ordered! I ordered 9 black remotes and they sent me ugly cream co...


In [ ]:
# ============================================================================
# STEP 5: Configure Training
# ============================================================================

training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/llama-profanity-qlora",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch size = 2 * 8 = 16
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),  # Use bf16 if available
    fp16=not torch.cuda.is_bf16_supported(),  # Otherwise fp16
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    warmup_steps=10,
    report_to="none",  # Disable wandb
    dataset_text_field="text",  # Important: tell trainer which field has the text
    max_seq_length=2048,
)

In [ ]:
# ============================================================================
# STEP 6: Create Trainer
# ============================================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],  # Add validation set
    tokenizer=tokenizer,
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/1227 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"]:   0%|          | 0/137 [00:00<?, ? examples/s]

In [ ]:
# ============================================================================
# STEP 7: Train!
# ============================================================================

print("\n🚀 Starting training...")
trainer_stats = trainer.train()

print("\n✅ Training complete!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")

The model is already on multiple devices. Skipping the move to device specified in `args`.



🚀 Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,227 | Num Epochs = 3 | Total steps = 231
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
100,0.958600,1.116448
200,0.720800,1.222950


Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



✅ Training complete!
Training time: 2417.32 seconds


In [ ]:
# ============================================================================
# STEP 8: Save Model
# ============================================================================

# Save LoRA adapters
model.save_pretrained("/content/drive/MyDrive/llama-profanity-lora")
tokenizer.save_pretrained("/content/drive/MyDrive/llama-profanity-lora")

print("\n✅ Model saved to Google Drive!")


✅ Model saved to Google Drive!


In [ ]:
# ============================================================================
# INFERENCE CODE
# ============================================================================

# Enable fast inference
FastLanguageModel.for_inference(model)

def analyze_review(review_text):
    """Test your fine-tuned model"""
    instruction = f"""Analyze the following customer review and provide:
1. Whether it contains profanity (yes/no)
2. The sentiment (positive/negative/neutral)
3. If profanity is present, rewrite it in a polite, professional manner while preserving the meaning and sentiment.

Review: {review_text}"""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

{instruction}

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        min_new_tokens=50,
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=False)

    # Extract assistant response
    response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()

    return response

# Test
test_reviews = [
    "This phone is damn slow",
    "Amazing product!",
    "Worst shit ever"
]

for review in test_reviews:
    print(f"\n{'='*60}")
    print(f"Review: {review}")
    print(f"{'='*60}")
    print(analyze_review(review))


Review: This phone is damn slow
Profanity: yes
Sentiment: negative
Rewritten:

Review: Amazing product!
Profanity: no | Sentiment: positive | Rewritten:

Review: Worst shit ever
Profanity: yes
Sentiment: negative
Rewritten:


In [ ]:
from unsloth import FastLanguageModel

# Load BASE model (no fine-tuning)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="LiquidAI/LFM2.5-1.2B-Instruct",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

def analyze_review_few_shot(review_text):
    """Use few-shot prompting instead of fine-tuning"""

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

You are a review analysis system. For each review, you must:
1. Detect if it contains profanity (yes/no)
2. Determine sentiment (positive/negative/neutral)
3. If profane, rewrite it professionally while keeping the meaning

Examples:

Review: "This damn phone is so slow"
Analysis:
Profanity: yes
Sentiment: negative
Rewritten: This phone is very slow

Review: "The battery life is shit"
Analysis:
Profanity: yes
Sentiment: negative
Rewritten: The battery life is poor

Review: "Love this product!"
Analysis:
Profanity: no
Sentiment: positive
Rewritten: Not needed

Now analyze this review:

Review: {review_text}
Analysis:

<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = response.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()

    return response

# Test it
test_reviews = [
    "This phone is damn slow",
    "Worst shit ever",
    "Amazing product!",
]

print("="*70)
print("TESTING BASE MODEL WITH FEW-SHOT PROMPTING")
print("="*70)

for review in test_reviews:
    print(f"\nReview: {review}")
    print("-"*70)
    print(analyze_review_few_shot(review))
    print("="*70)

==((====))==  Unsloth 2026.1.2: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
TESTING BASE MODEL WITH FEW-SHOT PROMPTING

Review: This phone is damn slow
----------------------------------------------------------------------
Detected Profanity: Yes  
Sentiment Analysis: Negative  
Professional Rewrite: The device performs sluggishly and may require more frequent charging."

</|end_header id|>><|im_end|>

Review: Worst shit ever
----------------------------------------------------------------------
I'm analyzing this customer feedback and need to classify its sentiment and check for profanity. If profanity is present, I will